In [ ]:
#========================================================================
# Name: filter_wrf_dsd_properties_congestus.ipynb
# Author: McKenna W. Stanford
# Author Contact: mckenna.stanford@pnnl.gov
#
# Description: Filter WRF DSD properties for congestus clouds and calculate
#              derived microphysical parameters for 3km CACTI domain.
#========================================================================

In [1]:
#===============================
# Imports
#===============================
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import glob
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle, Patch
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec
import datetime
import matplotlib.colors as colors
import matplotlib as mpl
from pytz import utc
from matplotlib.lines import Line2D
import pickle
from scipy.spatial import cKDTree

import dask
from distributed import Client, LocalCluster
from dask import delayed, compute
import dask.array as da

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import cartopy.io.img_tiles as cimgt
import shapely.geometry as sgeom
import matplotlib.patheffects as path_effects

import warnings
warnings.filterwarnings("ignore")

plt.rcParams['text.usetex'] = True

/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()


In [2]:
path = '/pscratch/sd/m/mckenna/cacti/wrf/'
ct_files = sorted(glob.glob(path+'derived_3km/*.nc'))
dsd_files = sorted(glob.glob(path+'derived_3km_dsd_parameters/*.nc'))
num_ct_files = len(ct_files)
num_dsd_files = len(dsd_files)
print('# of cloud top files:',num_ct_files)
print('# of DSD files:',num_dsd_files)

# of cloud top files: 8794
# of DSD files: 8794


In [3]:
ds = xr.open_dataset(ct_files[3600])
print(np.nanmax(ds['cth_tau_ir'].values))
print(np.nanmin(ds['cth_tau_ir'].values))

4.487443359375
0.9868469848632813


# Need to get 3D WRF files as well, since I didn't save temperature

In [4]:
read_path = '/pscratch/sd/m/mckenna/'
file_name = read_path+'wrf_matched_file_lists.p'
with open(file_name, 'rb') as f:
    wrf_files_dict = pickle.load(f)
wrf_3d_files = wrf_files_dict['wrf_3d_files']
wrf_3d_datetimes = wrf_files_dict['wrf_3d_datetimes']

In [5]:
def subset_var_2d(data,mask):

    # mask values where the csapr mask == 0.
    masked_data = np.ma.masked_array(data, mask=mask == 0)

    # Extract the 2D subset defined by the mask
    # The unmasked elements will define the dimensions
    unmasked_data = masked_data[~masked_data.mask]  # Extract non-masked data


    row_indices, col_indices = np.where(~masked_data.mask)  # Get the unmasked indices

    new_rows = len(np.unique(row_indices))  # Number of unique rows in the unmasked region
    new_cols = len(np.unique(col_indices))  # Number of unique columns in the unmasked region

    reshaped_data = unmasked_data.reshape((new_rows, new_cols))

    return reshaped_data.data

In [6]:
def subset_var_3d(data,mask):

    nz = len(data[:,0,0])
    # mask values where the csapr mask == 0.
    reshaped_data_3d = []
    
    for kk in range(nz):
        data_2d = data[kk,:,:]
        masked_data = np.ma.masked_array(data_2d, mask=mask == 0)
        #print(aaaaa)
        
        # Extract the 2D subset defined by the mask
        # The unmasked elements will define the dimensions
        unmasked_data = masked_data[~masked_data.mask]  # Extract non-masked data
        
        
        row_indices, col_indices = np.where(~masked_data.mask)  # Get the unmasked indices
        
        new_rows = len(np.unique(row_indices))  # Number of unique rows in the unmasked region
        new_cols = len(np.unique(col_indices))  # Number of unique columns in the unmasked region
        
        reshaped_data = unmasked_data.reshape((new_rows, new_cols))
    
        reshaped_data_3d.append(reshaped_data.data)

    reshaped_data_3d = np.asarray(reshaped_data_3d)
    
    return reshaped_data_3d

In [7]:
def get_csapr_mask(wrf_ref_file, wrf_2d_file):
    # Open datasets
    ds_ref = xr.open_dataset(wrf_ref_file)
    lon_ref = ds_ref['XLONG'].squeeze().values
    lat_ref = ds_ref['XLAT'].squeeze().values
    ds_ref.close()

    ds_2d = xr.open_dataset(wrf_2d_file)
    lon_2d = ds_2d['CAC_LONG'].squeeze().values
    lat_2d = ds_2d['CAC_LAT'].squeeze().values
    ds_2d.close()

    # Flatten the 2D lat/lon arrays into a list of (lon, lat) points
    ref_points = np.column_stack((lon_ref.ravel(), lat_ref.ravel()))
    wrf_points = np.column_stack((lon_2d.ravel(), lat_2d.ravel()))

    # Build a KDTree from the reference WRF grid (fast spatial lookup)
    tree = cKDTree(ref_points)

    # Query the WRF 2D grid against the reference grid
    _, indices = tree.query(wrf_points, distance_upper_bound=1e-6)  # Use a small tolerance

    # Initialize mask with zeros
    csapr_mask = np.zeros_like(lon_2d, dtype=float)

    # Only mark valid matches (ignore out-of-bounds results)
    valid_matches = indices < ref_points.shape[0]
    csapr_mask.ravel()[valid_matches] = 1.0

    return csapr_mask,lon_2d,lat_2d


In [8]:
def get_terrain_height(csapr_mask):
    ter_path = '/global/cfs/projectdirs/m1657/avarble/cacti/wrf/output/2019-05-01/'
    ter_file = ter_path+'wrfout_d01_2019-05-01_00:00:00'
    ds_ter = xr.open_dataset(ter_file,engine='netcdf4')
    ter_hgt = ds_ter['HGT'].squeeze()
    ds_ter.close()
    ter_hgt = subset_var_2d(ter_hgt,csapr_mask)

    return ter_hgt

In [9]:
def trim_var(lon,lat,var,var_dim):
    lonmin = -64.95
    lonmax = -64.65
    latmin = -32.9
    latmax = -31.1
    
    mean_lon = np.mean(lon,axis=0)
    mean_lat = np.mean(lat,axis=1)  
    
    # Longitude
    trim_id = np.where( (mean_lon >= lonmin) & (mean_lon <= lonmax)  )[0]
    if var_dim == 2:
        var = var[:,trim_id]
    elif var_dim == 3:
        var = var[:,:,trim_id]

    # Latitude
    trim_id = np.where( (mean_lat >= latmin) & (mean_lat <= latmax)  )[0]
    if var_dim == 2:
        var = var[trim_id,:]
    elif var_dim == 3:
        var = var[:,trim_id,:]
        
    return var

# Get some static variables that are only needed once

In [10]:
#=======================
# Get height from different file
#=======================
read_path = '/pscratch/sd/m/mckenna/'
file_name = read_path+'wrf_matched_file_lists.p'
with open(file_name, 'rb') as f:
    wrf_files_dict = pickle.load(f)
    wrf_ref_files = wrf_files_dict['wrf_ref_files']
    wrf_2d_files = wrf_files_dict['wrf_2d_files']
ds_ref = xr.open_dataset(wrf_ref_files[0])
zamsl = ds_ref['ZAMSL'].values.squeeze()
ref = ds_ref['REFL_10CM'].values.squeeze()
col_max_ref = np.nanmax(ref,axis=0)
csapr_nan_mask = np.isnan(col_max_ref)
ds_ref.close()

nz,ny,nx = zamsl.shape
z_agl = np.zeros((nz,ny,nx))

csapr_mask,lon_2d,lat_2d = get_csapr_mask(wrf_ref_files[0],wrf_2d_files[0])
ter_hgt = get_terrain_height(csapr_mask)
for kk in range(nz):
    z_agl[kk,:,:] = zamsl[kk,:,:] - ter_hgt

np.savez("static_vars.npz",z_agl=z_agl,zamsl=zamsl,ter_hgt=ter_hgt,lon_2d=lon_2d,lat_2d=lat_2d,csapr_mask=csapr_mask,csapr_nan_mask=csapr_nan_mask)

# Serial Run

In [11]:
#for ii in range(3100,num_ct_files):
#for ii in range(3650,num_ct_files):
#for ii in range(3660,num_ct_files):

#for ii in range(num_ct_files):
for ii in range(3600,3601):

    ct_file = ct_files[ii]
    dsd_file = dsd_files[ii]
    print('CT file:',ct_file)
    print('DSD file:',dsd_file)
    ct_str = ct_file.split('/')[-1].split('.')[-2].split('_')[-2:]
    ct_date_str = ct_str[0]
    ct_time_str = ct_str[1].split(':')
    ct_year = int(ct_date_str[0:4])
    ct_month = int(ct_date_str[4:6])
    ct_day = int(ct_date_str[6:8])
    ct_hour = int(ct_time_str[0])
    ct_min = int(ct_time_str[1])
    ct_sec = int(ct_time_str[2])
    ct_datetime = datetime.datetime(ct_year,ct_month,ct_day,ct_hour,ct_min,ct_sec,tzinfo=datetime.timezone.utc)

    dsd_str = dsd_file.split('/')[-1].split('.')[-2].split('_')[-2:]
    dsd_date_str = dsd_str[0]
    dsd_time_str = dsd_str[1].split(':')
    dsd_year = int(dsd_date_str[0:4])
    dsd_month = int(dsd_date_str[4:6])
    dsd_day = int(dsd_date_str[6:8])
    dsd_hour = int(dsd_time_str[0])
    dsd_min = int(dsd_time_str[1])
    dsd_sec = int(dsd_time_str[2])
    dsd_datetime = datetime.datetime(dsd_year,dsd_month,dsd_day,dsd_hour,dsd_min,dsd_sec,tzinfo=datetime.timezone.utc)
    print('CT datetime:',ct_datetime)
    print('DSD datetime:',dsd_datetime)
    #print('Time:',dsd_datetime)

    if ct_datetime != dsd_datetime:
        raise RuntimeError('Times between cloud-top file and DSD file do not match...')


    static = np.load("static_vars.npz")
    z_agl = static["z_agl"]
    csapr_nan_mask = static['csapr_nan_mask']
    #zamsl = static["zamsl"]
    #csapr_mask = static["csapr_mask"]
    #lon_2d = static["lon_2d"]
    #lat_2d = static["lat_2d"]
    
    
    ds_ct = xr.open_dataset(ct_file)
    cth = ds_ct['cth_tau_ir'].values 
    cth = cth + (ter_hgt*1.e-3) # CTH was calculated using height AGL, but actually needs to be AMSL, so adding it back here
    #==============================================================================
    # Skip file entirely if entirely absent of cloud
    #==============================================================================
    if np.isnan(np.nanmax(cth)):
        ds_ct.close()
        print('Skipping file because no cloud present.')
        continue
    ctt = ds_ct['ctt_tau_ir'] 
    #==============================================================================
    # Skip file if the warmest CTT in the entire domain is colder than -25 deg C.
    #==============================================================================
    if np.nanmax(ctt) < -25.:
        ds_ct.close()
        print('Skipping file because warmest CTT is colder than -25 deg C.')
        continue
    lon = ds_ct['lon']
    lat = ds_ct['lat']
    ctt_dim = 2
    ctt_trim = trim_var(lon,lat,ctt,2)
    lon_trim = trim_var(lon,lat,lon,2)
    lat_trim = trim_var(lon,lat,lat,2)
    #==============================================================================
    # Skip file if warmest CTT in the trimmed domain is colder than -25 deg C.
    #==============================================================================
    if (np.nanmax(ctt_trim) < -25.) or (np.isnan(np.nanmax(ctt_trim))):
        ds_ct.close()
        print('Skipping file because warmest CTT in trimmed domain is colder than -25 deg C or is NaN.')
        continue
        
    # Read in other variables to be used for filtering
    lwp = ds_ct['lwp']
    twp = ds_ct['twp']
    iwp = ds_ct['iwp']
    #opd = ds_ct['opd_combined']
    opd = ds_ct['opd_vis']

    # Trim them
    lwp_trim = trim_var(lon,lat,lwp,2)
    twp_trim = trim_var(lon,lat,twp,2)
    iwp_trim = trim_var(lon,lat,iwp,2)
    opd_trim = trim_var(lon,lat,opd,2)
    cth_trim = trim_var(lon,lat,cth,2)
    ds_ct.close()

    #==============================================================================
    #==============================================================================
    # Now extract DSD parameters
    #==============================================================================
    #==============================================================================
    ds_dsd = xr.open_dataset(dsd_file)
    D_num = ds_dsd['D_num'] # m
    D_vol = ds_dsd['D_vol'] # m
    D_mass = ds_dsd['D_mass'] # m
    mmd = ds_dsd['mmd'] # m
    N = ds_dsd['N'] # kg^-1
    q = ds_dsd['q'] # kg/kg
    rho_air = ds_dsd['rho_air'] # kg/m^3
    #sigma_num = ds_dsd['sigma_num'] # m
    #skew_num = ds_dsd['skew_num'] # unitless
    #kurt_num = ds_dsd['kurt_num'] # unitless
    #sigma_mass = ds_dsd['sigma_mass'] # m
    #skew_mass = ds_dsd['skew_mass'] # unitless
    #kurt_mass = ds_dsd['kurt_mass'] # unitless
    N_0_r = ds_dsd['N_0_r'] # m^-4
    N_0_c = ds_dsd['N_0_c'] # m^-4
    lambda_r = ds_dsd['lambda_r'] # m^-1
    lambda_c = ds_dsd['lambda_c'] # m^-1
    mu_c = ds_dsd['mu_c'] # unitless
    q_gt_100um = ds_dsd['q_gt_100um'] # kg/kg
    N_gt_100um = ds_dsd['N_gt_100um'] # kg^-1
    ds_dsd.close()

    # NaN-out values that lie outside the CSAPR range
    D_num = D_num.where(~csapr_nan_mask)
    D_vol = D_vol.where(~csapr_nan_mask)
    D_mass = D_mass.where(~csapr_nan_mask)
    mmd = mmd.where(~csapr_nan_mask)
    N = N.where(~csapr_nan_mask)
    q = q.where(~csapr_nan_mask)
    rho_air = rho_air.where(~csapr_nan_mask)
    #sigma_num = sigma_num.where(~csapr_nan_mask)
    #skew_num = skew_num.where(~csapr_nan_mask)
    #kurt_num = kurt_num.where(~csapr_nan_mask)
    #sigma_mass = sigma_mass.where(~csapr_nan_mask)
    #skew_mass = skew_mass.where(~csapr_nan_mask)
    #kurt_mass = kurt_mass.where(~csapr_nan_mask)
    N_0_r = N_0_r.where(~csapr_nan_mask)
    N_0_c = N_0_c.where(~csapr_nan_mask)
    lambda_r = lambda_r.where(~csapr_nan_mask)
    lambda_c = lambda_c.where(~csapr_nan_mask)
    mu_c = mu_c.where(~csapr_nan_mask)
    q_gt_100um = q_gt_100um.where(~csapr_nan_mask)
    N_gt_100um = N_gt_100um.where(~csapr_nan_mask)

    D_num_trim = trim_var(lon,lat,D_num,3)
    D_vol_trim = trim_var(lon,lat,D_vol,3)
    D_mass_trim = trim_var(lon,lat,D_mass,3)
    mmd_trim = trim_var(lon,lat,mmd,3)
    N_trim = trim_var(lon,lat,N,3)
    q_trim = trim_var(lon,lat,q,3)
    rho_air_trim = trim_var(lon,lat,rho_air,3)
    #sigma_num_trim = trim_var(lon,lat,sigma_num,3)
    #skew_num_trim = trim_var(lon,lat,skew_num,3)
    #kurt_num_trim = trim_var(lon,lat,kurt_num,3)
    #sigma_mass_trim = trim_var(lon,lat,sigma_mass,3)
    #skew_mass_trim = trim_var(lon,lat,skew_mass,3)
    #kurt_mass_trim = trim_var(lon,lat,kurt_mass,3)
    N_0_r_trim = trim_var(lon,lat,N_0_r,3)
    N_0_c_trim = trim_var(lon,lat,N_0_c,3)
    lambda_r_trim = trim_var(lon,lat,lambda_r,3)
    lambda_c_trim = trim_var(lon,lat,lambda_c,3)
    mu_c_trim = trim_var(lon,lat,mu_c,3)
    q_gt_100um_trim = trim_var(lon,lat,q_gt_100um,3)
    N_gt_100um_trim = trim_var(lon,lat,N_gt_100um,3)

    #------------------------------------------
    # Get temperature
    # Also get Qr, Qc. Don't actually need these,
    # but they're helpful conceptually during testing.
    #------------------------------------------
    time_id = np.where(wrf_3d_datetimes == ct_datetime)
    if np.size(time_id) > 0.:
        time_id = time_id[0][0]
    else:
        raise RuntimeError('No matchting WRF 3D file.')
    ds_3d = xr.open_dataset(wrf_3d_files[time_id])
    temp = ds_3d['CAC_T'].squeeze() # K
    qr = ds_3d['CAC_QR'].squeeze()*1.e-3 # kg/kg
    qc = ds_3d['CAC_QC'].squeeze()*1.e-3 # kg/kg
    Nr = ds_3d['CAC_NR'].squeeze()*1.e-3 # kg/kg
    Nc = ds_3d['CAC_NC'].squeeze()*1.e-3 # kg/kg
    qv = ds_3d['CAC_QV'].squeeze()*1.e-3 # kg/kg
    pres = ds_3d['CAC_P'].squeeze() # Pa
    ds_3d.close()

    # Calculate RH
    C0 = 0.611583699e3
    C1 = 0.444606896e2
    C2 = 0.143177157e1
    C3 = 0.264224321e-1
    C4 = 0.299291081e-3
    C5 = 0.203154182e-5
    C6 = 0.702620698e-8
    C7 = 0.379534310e-11
    C8 = -0.321582393e-13
    X = temp.values - 273.15 # temperature in deg C
    ESL = C0 + X*(C1 + X*(C2 + X*(C3 + X*(C4 + X*(C5 + X*(C6 + X*(C7 + X*C8))))))) #saturation vapor pressure
    QVS = 0.622*ESL/(pres.values - ESL) #saturation vapor mixing ratio
    RH = 1e2*qv.values/QVS # %

    
    temp = subset_var_3d(temp,csapr_mask)
    RH = subset_var_3d(RH,csapr_mask)
    qr = subset_var_3d(qr,csapr_mask)
    qc = subset_var_3d(qc,csapr_mask)
    Nr = subset_var_3d(Nr,csapr_mask)
    Nc = subset_var_3d(Nc,csapr_mask)
    qr[:,csapr_nan_mask] = np.nan
    Nr[:,csapr_nan_mask] = np.nan
    qc[:,csapr_nan_mask] = np.nan
    Nc[:,csapr_nan_mask] = np.nan
    
    temp_trim = trim_var(lon,lat,temp,3)-273.15
    RH_trim = trim_var(lon,lat,RH,3)
    qr_trim = trim_var(lon,lat,qr,3)
    Nr_trim = trim_var(lon,lat,Nr,3)
    qc_trim = trim_var(lon,lat,qc,3)
    Nc_trim = trim_var(lon,lat,Nc,3)
    zamsl_trim = trim_var(lon,lat,zamsl,3)
    z_agl_trim = trim_var(lon,lat,z_agl,3)


    #------------------------------------------
    # Limit profiles to be under 10 km, as a start to limit memory
    #------------------------------------------
    zamsl_mean = np.mean(zamsl_trim,axis=(1,2))
    temp_mean = np.mean(temp_trim,axis=(1,2))
    height_id = np.where(zamsl_mean < 10000.)[0]

    temp_trim = temp_trim[height_id,:,:]
    RH_trim = RH_trim[height_id,:,:]
    qr_trim = qr_trim[height_id,:,:]
    Nr_trim = Nr_trim[height_id,:,:]
    qc_trim = qc_trim[height_id,:,:]
    Nc_trim = Nc_trim[height_id,:,:]
    zamsl_trim = zamsl_trim[height_id,:,:]
    z_agl_trim = z_agl_trim[height_id,:,:]
    D_num_trim = D_num_trim[height_id,:,:]
    D_vol_trim = D_vol_trim[height_id,:,:]
    D_mass_trim = D_mass_trim[height_id,:,:]
    mmd_trim = mmd_trim[height_id,:,:]
    N_trim = N_trim[height_id,:,:]
    q_trim = q_trim[height_id,:,:]
    rho_air_trim = rho_air_trim[height_id,:,:]
    #sigma_num_trim = sigma_num_trim[height_id,:,:]
    #skew_num_trim = skew_num_trim[height_id,:,:]
    #kurt_num_trim = kurt_num_trim[height_id,:,:]
    #sigma_mass_trim = sigma_mass_trim[height_id,:,:]
    #skew_mass_trim = skew_mass_trim[height_id,:,:]
    #kurt_mass_trim = kurt_mass_trim[height_id,:,:]
    N_0_r_trim = N_0_r_trim[height_id,:,:]
    N_0_c_trim = N_0_c_trim[height_id,:,:]
    lambda_r_trim = lambda_r_trim[height_id,:,:]
    lambda_c_trim = lambda_c_trim[height_id,:,:]
    mu_c_trim = mu_c_trim[height_id,:,:]
    q_gt_100um_trim = q_gt_100um_trim[height_id,:,:]
    N_gt_100um_trim = N_gt_100um_trim[height_id,:,:]


    #------------------------------------------
    # Identify profiles where CTT > -25 deg C to isolate
    # cumulus and congestus profiles. Resulting arrays will
    # be of shape nz x num_cong_profiles
    #------------------------------------------ 
    nz_trim = len(zamsl_trim[:,0,0])
    #cong_mask = ctt_trim >= -25.
    cong_mask = (ctt_trim >= -20.) & (cth_trim >= 0.5) & (opd_trim > 10.)

    flat_mask = cong_mask.values.flatten()
    zamsl_flat = zamsl_trim.reshape(nz_trim, -1)  # Shape: (47, 66×9) = (47, 594)  
    zamsl_filtered = zamsl_flat[:, flat_mask]  # Shape: (47, X)
    z_agl_flat = z_agl_trim.reshape(nz_trim, -1)
    z_agl_filtered = z_agl_flat[:, flat_mask]
    temp_flat = temp_trim.reshape(nz_trim, -1)
    temp_filtered = temp_flat[:, flat_mask]
    RH_flat = RH_trim.reshape(nz_trim, -1)
    RH_filtered = RH_flat[:, flat_mask]
    qr_flat = qr_trim.reshape(nz_trim, -1)
    qr_filtered = qr_flat[:, flat_mask]
    qc_flat = qc_trim.reshape(nz_trim, -1)
    qc_filtered = qc_flat[:, flat_mask]
    Nr_flat = Nr_trim.reshape(nz_trim, -1)
    Nr_filtered = Nr_flat[:, flat_mask]
    Nc_flat = Nc_trim.reshape(nz_trim, -1)
    Nc_filtered = Nc_flat[:, flat_mask]
    
    D_num_flat = D_num_trim.values.reshape(nz_trim, -1)
    D_num_filtered = D_num_flat[:, flat_mask]
    D_vol_flat = D_vol_trim.values.reshape(nz_trim, -1)
    D_vol_filtered = D_vol_flat[:, flat_mask]
    D_mass_flat = D_mass_trim.values.reshape(nz_trim, -1)
    D_mass_filtered = D_mass_flat[:, flat_mask]
    mmd_flat = mmd_trim.values.reshape(nz_trim, -1)
    mmd_filtered = mmd_flat[:, flat_mask]
    N_flat = N_trim.values.reshape(nz_trim, -1)
    N_filtered = N_flat[:, flat_mask]
    q_flat = q_trim.values.reshape(nz_trim, -1)
    q_filtered = q_flat[:, flat_mask]
    rho_air_flat = rho_air_trim.values.reshape(nz_trim, -1)
    rho_air_filtered = rho_air_flat[:, flat_mask]
    #sigma_num_flat = sigma_num_trim.values.reshape(nz_trim, -1)
    #sigma_num_filtered = sigma_num_flat[:, flat_mask]
    #skew_num_flat = skew_num_trim.values.reshape(nz_trim, -1)
    #skew_num_filtered = skew_num_flat[:, flat_mask]
    #kurt_num_flat = kurt_num_trim.values.reshape(nz_trim, -1)
    #kurt_num_filtered = kurt_num_flat[:, flat_mask]
    #sigma_mass_flat = sigma_mass_trim.values.reshape(nz_trim, -1)
    #sigma_mass_filtered = sigma_mass_flat[:, flat_mask]
    #skew_mass_flat = skew_mass_trim.values.reshape(nz_trim, -1)
    #skew_mass_filtered = skew_mass_flat[:, flat_mask]
    #kurt_mass_flat = kurt_mass_trim.values.reshape(nz_trim, -1)
    #kurt_mass_filtered = kurt_mass_flat[:, flat_mask]
    N_0_r_flat = N_0_r_trim.values.reshape(nz_trim, -1)
    N_0_r_filtered = N_0_r_flat[:, flat_mask]
    N_0_c_flat = N_0_c_trim.values.reshape(nz_trim, -1)
    N_0_c_filtered = N_0_c_flat[:, flat_mask]
    lambda_r_flat = lambda_r_trim.values.reshape(nz_trim, -1)
    lambda_r_filtered = lambda_r_flat[:, flat_mask]
    lambda_c_flat = lambda_c_trim.values.reshape(nz_trim, -1)
    lambda_c_filtered = lambda_c_flat[:, flat_mask]
    mu_c_flat = mu_c_trim.values.reshape(nz_trim, -1)
    mu_c_filtered = mu_c_flat[:, flat_mask]
    q_gt_100um_flat = q_gt_100um_trim.values.reshape(nz_trim, -1)
    q_gt_100um_filtered = q_gt_100um_flat[:, flat_mask]
    N_gt_100um_flat = N_gt_100um_trim.values.reshape(nz_trim, -1)
    N_gt_100um_filtered = N_gt_100um_flat[:, flat_mask]

    ctt_flat = ctt_trim.values.flatten()
    ctt_filtered = ctt_flat[flat_mask]  # Shape: (X,)
    cth_flat = cth_trim.flatten()
    cth_filtered = cth_flat[flat_mask]  
    opd_flat = opd_trim.values.flatten()
    opd_filtered = opd_flat[flat_mask]  
    lwp_flat = lwp_trim.values.flatten()
    lwp_filtered = lwp_flat[flat_mask]  
    twp_flat = twp_trim.values.flatten()
    twp_filtered = twp_flat[flat_mask] 
    iwp_flat = iwp_trim.values.flatten()
    iwp_filtered = iwp_flat[flat_mask] 
    
    #---------------------------------------------------
    # Now limit to points w/ total LWC > 1.e-3 g/m^3
    #---------------------------------------------------
    lwc_filtered = q_filtered*rho_air_filtered*1.e3 # g/m^3
    #valid_mask = (lwc_filtered > 1.e-3) & (temp_filtered >= 0.)
    valid_mask = (lwc_filtered > 1.e-3)
    if np.size(valid_mask) == 0.:
        print('No congestus ID points w/ LWC > 1.e-3 g/m^3.')
        continue

    lwc_valid = lwc_filtered[valid_mask]
    temp_valid = temp_filtered[valid_mask]
    RH_valid = RH_filtered[valid_mask]
    D_num_valid = D_num_filtered[valid_mask]
    D_vol_valid = D_vol_filtered[valid_mask]
    D_mass_valid = D_mass_filtered[valid_mask]
    mmd_valid = mmd_filtered[valid_mask]
    N_valid = N_filtered[valid_mask]
    q_valid = q_filtered[valid_mask]
    rho_air_valid = rho_air_filtered[valid_mask]
    #sigma_num_valid = sigma_num_filtered[valid_mask]
    #skew_num_valid = skew_num_filtered[valid_mask]
    #kurt_num_valid = kurt_num_filtered[valid_mask]
    #sigma_mass_valid = sigma_mass_filtered[valid_mask]
    #skew_mass_valid = skew_mass_filtered[valid_mask]
    #kurt_mass_valid = kurt_mass_filtered[valid_mask]
    N_0_r_valid = N_0_r_filtered[valid_mask]
    N_0_c_valid = N_0_c_filtered[valid_mask]
    lambda_r_valid = lambda_r_filtered[valid_mask]
    lambda_c_valid = lambda_c_filtered[valid_mask]
    mu_c_valid = mu_c_filtered[valid_mask]
    q_gt_100um_valid = q_gt_100um_filtered[valid_mask]
    N_gt_100um_valid = N_gt_100um_filtered[valid_mask]
    qr_valid = qr_filtered[valid_mask]
    Nr_valid = Nr_filtered[valid_mask]
    qc_valid = qc_filtered[valid_mask]
    Nc_valid = Nc_filtered[valid_mask]
    z_agl_valid = z_agl_filtered[valid_mask]
    zamsl_valid = zamsl_filtered[valid_mask]
    
    save_path = '/pscratch/sd/m/mckenna/cacti/wrf/temp_cong_dsd_params/'
    str_dt = dsd_datetime.strftime('%Y%m%d_%H:%M:%S')
    np.savez(save_path+"cong_dsd_params_"+str_dt+"_v2.npz",\
                            z_agl=z_agl_valid,\
                            zamsl=zamsl_valid,\
                            temp=temp_valid,\
                            RH=RH_valid,\
                            lwc=lwc_valid,\
                            D_num=D_num_valid,\
                            D_vol=D_vol_valid,\
                            D_mass=D_mass_valid,\
                            mmd=mmd_valid,\
                            N=N_valid,\
                            q=q_valid,\
                            rho_air=rho_air_valid,\
                            #sigma_num=sigma_num_valid,\
                            #skew_num=skew_num_valid,\
                            #kurt_num=kurt_num_valid,\
                            #sigma_mass=sigma_mass_valid,\
                            #skew_mass=skew_mass_valid,\
                            #kurt_mass=kurt_mass_valid,\
                            N_0_r=N_0_r_valid,\
                            N_0_c=N_0_c_valid,\
                            lambda_r=lambda_r_valid,\
                            lambda_c=lambda_c_valid,\
                            mu_c=mu_c_valid,\
                            q_gt_100um=q_gt_100um_valid,\
                            N_gt_100um=N_gt_100um_valid,\
                            qr=qr_valid,\
                            qc=qc_valid,\
                            Nr=Nr_valid,\
                            Nc=Nc_valid,\
                            )
    #break
    #print(aaaaaaa)

    print('')
print('done')

CT file: /pscratch/sd/m/mckenna/cacti/wrf/derived_3km/WRF_CACTI_3km_derived_20181123_04:45:00.nc
DSD file: /pscratch/sd/m/mckenna/cacti/wrf/derived_3km_dsd_parameters/WRF_CACTI_3km_DSD_parameters_liq_only_20181123_04:45:00.nc
CT datetime: 2018-11-23 04:45:00+00:00
DSD datetime: 2018-11-23 04:45:00+00:00

done


In [18]:
type(opd_trim)

xarray.core.dataarray.DataArray

In [13]:
print(np.shape(temp.values),np.max(temp.values),np.min(temp.values))
print(np.shape(pres.values),np.max(pres.values),np.min(pres.values))
print(np.shape(qv.values),np.max(qv.values),np.min(qv.values))

(80, 499, 599) 298.98404 -999.0
(80, 499, 599) 99844.6 -999.0
(80, 499, 599) 0.010867245 -0.9990001


In [11]:
C0 = 0.611583699e3
C1 = 0.444606896e2
C2 = 0.143177157e1
C3 = 0.264224321e-1
C4 = 0.299291081e-3
C5 = 0.203154182e-5
C6 = 0.702620698e-8
C7 = 0.379534310e-11
C8 = -0.321582393e-13
X = temp.values - 273.16 # temperature in deg C
ESL = C0 + X*(C1 + X*(C2 + X*(C3 + X*(C4 + X*(C5 + X*(C6 + X*(C7 + X*C8))))))) #saturation vapor pressure
QVS = 0.622*ESL/(pres.values - ESL) #saturation vapor mixing ratio
RH = 1e2*qv.values/QVS # %

NameError: name 'temp' is not defined

In [19]:
np.min(RH)

0.20952246

In [ ]:
print(np.nanmax(ctt_trim.values))
print(np.nanmin(ctt_trim.values))
print(np.nanmax(opd_trim.values))
print(np.nanmin(opd_trim.values))
print(np.nanmax(cth_trim.values))
print(np.nanmin(cth_trim.values))

In [11]:
#for ii in range(3100,num_ct_files):
#for ii in range(3650,num_ct_files):
#for ii in range(3660,num_ct_files):

#for ii in range(num_ct_files):
#for ii in range(3600,3700):
def driver_func(ct_file,dsd_file,wrf_3d_files):

    #ct_file = ct_files[ii]
    #dsd_file = dsd_files[ii]
    print('CT file:',ct_file)
    print('DSD file:',dsd_file)
    ct_str = ct_file.split('/')[-1].split('.')[-2].split('_')[-2:]
    ct_date_str = ct_str[0]
    ct_time_str = ct_str[1].split(':')
    ct_year = int(ct_date_str[0:4])
    ct_month = int(ct_date_str[4:6])
    ct_day = int(ct_date_str[6:8])
    ct_hour = int(ct_time_str[0])
    ct_min = int(ct_time_str[1])
    ct_sec = int(ct_time_str[2])
    ct_datetime = datetime.datetime(ct_year,ct_month,ct_day,ct_hour,ct_min,ct_sec,tzinfo=datetime.timezone.utc)
    
    dsd_str = dsd_file.split('/')[-1].split('.')[-2].split('_')[-2:]
    dsd_date_str = dsd_str[0]
    dsd_time_str = dsd_str[1].split(':')
    dsd_year = int(dsd_date_str[0:4])
    dsd_month = int(dsd_date_str[4:6])
    dsd_day = int(dsd_date_str[6:8])
    dsd_hour = int(dsd_time_str[0])
    dsd_min = int(dsd_time_str[1])
    dsd_sec = int(dsd_time_str[2])
    dsd_datetime = datetime.datetime(dsd_year,dsd_month,dsd_day,dsd_hour,dsd_min,dsd_sec,tzinfo=datetime.timezone.utc)
    #print('CT datetime:',ct_datetime)
    #print('DSD datetime:',dsd_datetime)
    #print('Time:',dsd_datetime)

    if ct_datetime != dsd_datetime:
        raise RuntimeError('Times between cloud-top file and DSD file do not match...')


    static = np.load("static_vars.npz")
    z_agl = static["z_agl"]
    csapr_nan_mask = static['csapr_nan_mask']
    #zamsl = static["zamsl"]
    #csapr_mask = static["csapr_mask"]
    #lon_2d = static["lon_2d"]
    #lat_2d = static["lat_2d"]
    
    
    ds_ct = xr.open_dataset(ct_file)
    cth = ds_ct['cth_tau_ir'].values 
    cth = cth + (ter_hgt*1.e-3) # CTH was calculated using height AGL, but actually needs to be AMSL, so adding it back here
    #==============================================================================
    # Skip file entirely if entirely absent of cloud
    #==============================================================================
    if np.isnan(np.nanmax(cth)):
        ds_ct.close()
        print('Skipping file because no cloud present.')
        return
    ctt = ds_ct['ctt_tau_ir'] 
    #==============================================================================
    # Skip file if the warmest CTT in the entire domain is colder than -25 deg C.
    #==============================================================================
    if np.nanmax(ctt) < -20.:
        ds_ct.close()
        print('Skipping file because warmest CTT is colder than -25 deg C.')
        return
    lon = ds_ct['lon']
    lat = ds_ct['lat']
    ctt_dim = 2
    ctt_trim = trim_var(lon,lat,ctt,2)
    lon_trim = trim_var(lon,lat,lon,2)
    lat_trim = trim_var(lon,lat,lat,2)
    #==============================================================================
    # Skip file if warmest CTT in the trimmed domain is colder than -25 deg C.
    #==============================================================================
    if (np.nanmax(ctt_trim) < -20.) or (np.isnan(np.nanmax(ctt_trim))):
        ds_ct.close()
        print('Skipping file because warmest CTT in trimmed domain is colder than -25 deg C or is NaN.')
        return
        
    # Read in other variables to be used for filtering
    lwp = ds_ct['lwp']
    twp = ds_ct['twp']
    iwp = ds_ct['iwp']
    opd = ds_ct['opd_vis']

    # Trim them
    lwp_trim = trim_var(lon,lat,lwp,2)
    twp_trim = trim_var(lon,lat,twp,2)
    iwp_trim = trim_var(lon,lat,iwp,2)
    opd_trim = trim_var(lon,lat,opd,2)
    cth_trim = trim_var(lon,lat,cth,2)
    ds_ct.close()

    #==============================================================================
    #==============================================================================
    # Now extract DSD parameters
    #==============================================================================
    #==============================================================================
    ds_dsd = xr.open_dataset(dsd_file)
    D_num = ds_dsd['D_num'] # m
    D_vol = ds_dsd['D_vol'] # m
    D_mass = ds_dsd['D_mass'] # m
    mmd = ds_dsd['mmd'] # m
    N = ds_dsd['N'] # kg^-1
    q = ds_dsd['q'] # kg/kg
    rho_air = ds_dsd['rho_air'] # kg/m^3
    #sigma_num = ds_dsd['sigma_num'] # m
    #skew_num = ds_dsd['skew_num'] # unitless
    #kurt_num = ds_dsd['kurt_num'] # unitless
    #sigma_mass = ds_dsd['sigma_mass'] # m
    #skew_mass = ds_dsd['skew_mass'] # unitless
    #kurt_mass = ds_dsd['kurt_mass'] # unitless
    N_0_r = ds_dsd['N_0_r'] # m^-4
    N_0_c = ds_dsd['N_0_c'] # m^-4
    lambda_r = ds_dsd['lambda_r'] # m^-1
    lambda_c = ds_dsd['lambda_c'] # m^-1
    mu_c = ds_dsd['mu_c'] # unitless
    q_gt_100um = ds_dsd['q_gt_100um'] # kg/kg
    N_gt_100um = ds_dsd['N_gt_100um'] # kg^-1
    ds_dsd.close()

    # NaN-out values that lie outside the CSAPR range
    D_num = D_num.where(~csapr_nan_mask)
    D_vol = D_vol.where(~csapr_nan_mask)
    D_mass = D_mass.where(~csapr_nan_mask)
    mmd = mmd.where(~csapr_nan_mask)
    N = N.where(~csapr_nan_mask)
    q = q.where(~csapr_nan_mask)
    rho_air = rho_air.where(~csapr_nan_mask)
    #sigma_num = sigma_num.where(~csapr_nan_mask)
    #skew_num = skew_num.where(~csapr_nan_mask)
    #kurt_num = kurt_num.where(~csapr_nan_mask)
    #sigma_mass = sigma_mass.where(~csapr_nan_mask)
    #skew_mass = skew_mass.where(~csapr_nan_mask)
    #kurt_mass = kurt_mass.where(~csapr_nan_mask)
    N_0_r = N_0_r.where(~csapr_nan_mask)
    N_0_c = N_0_c.where(~csapr_nan_mask)
    lambda_r = lambda_r.where(~csapr_nan_mask)
    lambda_c = lambda_c.where(~csapr_nan_mask)
    mu_c = mu_c.where(~csapr_nan_mask)
    q_gt_100um = q_gt_100um.where(~csapr_nan_mask)
    N_gt_100um = N_gt_100um.where(~csapr_nan_mask)

    D_num_trim = trim_var(lon,lat,D_num,3)
    D_vol_trim = trim_var(lon,lat,D_vol,3)
    D_mass_trim = trim_var(lon,lat,D_mass,3)
    mmd_trim = trim_var(lon,lat,mmd,3)
    N_trim = trim_var(lon,lat,N,3)
    q_trim = trim_var(lon,lat,q,3)
    rho_air_trim = trim_var(lon,lat,rho_air,3)
    #sigma_num_trim = trim_var(lon,lat,sigma_num,3)
    #skew_num_trim = trim_var(lon,lat,skew_num,3)
    #kurt_num_trim = trim_var(lon,lat,kurt_num,3)
    #sigma_mass_trim = trim_var(lon,lat,sigma_mass,3)
    #skew_mass_trim = trim_var(lon,lat,skew_mass,3)
    #kurt_mass_trim = trim_var(lon,lat,kurt_mass,3)
    N_0_r_trim = trim_var(lon,lat,N_0_r,3)
    N_0_c_trim = trim_var(lon,lat,N_0_c,3)
    lambda_r_trim = trim_var(lon,lat,lambda_r,3)
    lambda_c_trim = trim_var(lon,lat,lambda_c,3)
    mu_c_trim = trim_var(lon,lat,mu_c,3)
    q_gt_100um_trim = trim_var(lon,lat,q_gt_100um,3)
    N_gt_100um_trim = trim_var(lon,lat,N_gt_100um,3)

    #------------------------------------------
    # Get temperature
    # Also get Qr, Qc. Don't actually need these,
    # but they're helpful conceptually during testing.
    #------------------------------------------
    time_id = np.where(wrf_3d_datetimes == ct_datetime)
    if np.size(time_id) > 0.:
        time_id = time_id[0][0]
    else:
        raise RuntimeError('No matchting WRF 3D file.')
    ds_3d = xr.open_dataset(wrf_3d_files[time_id])
    temp = ds_3d['CAC_T'].squeeze() # K
    qr = ds_3d['CAC_QR'].squeeze()*1.e-3 # kg/kg
    qc = ds_3d['CAC_QC'].squeeze()*1.e-3 # kg/kg
    Nr = ds_3d['CAC_NR'].squeeze()*1.e-3 # /kg
    Nc = ds_3d['CAC_NC'].squeeze()*1.e-3 # /kg
    qv = ds_3d['CAC_QV'].squeeze()*1.e-3 # kg/kg
    pres = ds_3d['CAC_P'].squeeze() # Pa
    ds_3d.close()

    # Calculate RH
    C0 = 0.611583699e3
    C1 = 0.444606896e2
    C2 = 0.143177157e1
    C3 = 0.264224321e-1
    C4 = 0.299291081e-3
    C5 = 0.203154182e-5
    C6 = 0.702620698e-8
    C7 = 0.379534310e-11
    C8 = -0.321582393e-13
    X = temp.values - 273.15 # temperature in deg C
    ESL = C0 + X*(C1 + X*(C2 + X*(C3 + X*(C4 + X*(C5 + X*(C6 + X*(C7 + X*C8))))))) #saturation vapor pressure
    QVS = 0.622*ESL/(pres.values - ESL) #saturation vapor mixing ratio
    RH = 1e2*qv.values/QVS # %


    
    temp = subset_var_3d(temp,csapr_mask)
    RH = subset_var_3d(RH,csapr_mask)
    qr = subset_var_3d(qr,csapr_mask)
    qc = subset_var_3d(qc,csapr_mask)
    Nr = subset_var_3d(Nr,csapr_mask)
    Nc = subset_var_3d(Nc,csapr_mask)
    qr[:,csapr_nan_mask] = np.nan
    Nr[:,csapr_nan_mask] = np.nan
    qc[:,csapr_nan_mask] = np.nan
    Nc[:,csapr_nan_mask] = np.nan
    temp_trim = trim_var(lon,lat,temp,3)-273.15
    RH_trim = trim_var(lon,lat,RH,3)
    qr_trim = trim_var(lon,lat,qr,3)
    Nr_trim = trim_var(lon,lat,Nr,3)
    qc_trim = trim_var(lon,lat,qc,3)
    Nc_trim = trim_var(lon,lat,Nc,3)
    zamsl_trim = trim_var(lon,lat,zamsl,3)
    z_agl_trim = trim_var(lon,lat,z_agl,3)


    #------------------------------------------
    # Limit profiles to be under 10 km, as a start to limit memory
    #------------------------------------------
    zamsl_mean = np.mean(zamsl_trim,axis=(1,2))
    temp_mean = np.mean(temp_trim,axis=(1,2))
    height_id = np.where(zamsl_mean < 10000.)[0]

    temp_trim = temp_trim[height_id,:,:]
    RH_trim = RH_trim[height_id,:,:]
    qr_trim = qr_trim[height_id,:,:]
    Nr_trim = Nr_trim[height_id,:,:]
    qc_trim = qc_trim[height_id,:,:]
    Nc_trim = Nc_trim[height_id,:,:]
    zamsl_trim = zamsl_trim[height_id,:,:]
    z_agl_trim = z_agl_trim[height_id,:,:]
    D_num_trim = D_num_trim[height_id,:,:]
    D_vol_trim = D_vol_trim[height_id,:,:]
    D_mass_trim = D_mass_trim[height_id,:,:]
    mmd_trim = mmd_trim[height_id,:,:]
    N_trim = N_trim[height_id,:,:]
    q_trim = q_trim[height_id,:,:]
    rho_air_trim = rho_air_trim[height_id,:,:]
    #sigma_num_trim = sigma_num_trim[height_id,:,:]
    #skew_num_trim = skew_num_trim[height_id,:,:]
    #kurt_num_trim = kurt_num_trim[height_id,:,:]
    #sigma_mass_trim = sigma_mass_trim[height_id,:,:]
    #skew_mass_trim = skew_mass_trim[height_id,:,:]
    #kurt_mass_trim = kurt_mass_trim[height_id,:,:]
    N_0_r_trim = N_0_r_trim[height_id,:,:]
    N_0_c_trim = N_0_c_trim[height_id,:,:]
    lambda_r_trim = lambda_r_trim[height_id,:,:]
    lambda_c_trim = lambda_c_trim[height_id,:,:]
    mu_c_trim = mu_c_trim[height_id,:,:]
    q_gt_100um_trim = q_gt_100um_trim[height_id,:,:]
    N_gt_100um_trim = N_gt_100um_trim[height_id,:,:]


    #------------------------------------------
    # Identify profiles where CTT > -25 deg C to isolate
    # cumulus and congestus profiles. Resulting arrays will
    # be of shape nz x num_cong_profiles
    #------------------------------------------ 
    nz_trim = len(zamsl_trim[:,0,0])
    #cong_mask = ctt_trim >= -25.
    cong_mask = (ctt_trim >= -20.) & (cth_trim >= 0.5) & (opd_trim > 10.)

    flat_mask = cong_mask.values.flatten()
    zamsl_flat = zamsl_trim.reshape(nz_trim, -1)  # Shape: (47, 66×9) = (47, 594)  
    zamsl_filtered = zamsl_flat[:, flat_mask]  # Shape: (47, X)
    z_agl_flat = z_agl_trim.reshape(nz_trim, -1)
    z_agl_filtered = z_agl_flat[:, flat_mask]
    temp_flat = temp_trim.reshape(nz_trim, -1)
    temp_filtered = temp_flat[:, flat_mask]
    RH_flat = RH_trim.reshape(nz_trim, -1)
    RH_filtered = RH_flat[:, flat_mask]
    qr_flat = qr_trim.reshape(nz_trim, -1)
    qr_filtered = qr_flat[:, flat_mask]
    qc_flat = qc_trim.reshape(nz_trim, -1)
    qc_filtered = qc_flat[:, flat_mask]
    Nr_flat = Nr_trim.reshape(nz_trim, -1)
    Nr_filtered = Nr_flat[:, flat_mask]
    Nc_flat = Nc_trim.reshape(nz_trim, -1)
    Nc_filtered = Nc_flat[:, flat_mask]
    
    D_num_flat = D_num_trim.values.reshape(nz_trim, -1)
    D_num_filtered = D_num_flat[:, flat_mask]
    D_vol_flat = D_vol_trim.values.reshape(nz_trim, -1)
    D_vol_filtered = D_vol_flat[:, flat_mask]
    D_mass_flat = D_mass_trim.values.reshape(nz_trim, -1)
    D_mass_filtered = D_mass_flat[:, flat_mask]
    mmd_flat = mmd_trim.values.reshape(nz_trim, -1)
    mmd_filtered = mmd_flat[:, flat_mask]
    N_flat = N_trim.values.reshape(nz_trim, -1)
    N_filtered = N_flat[:, flat_mask]
    q_flat = q_trim.values.reshape(nz_trim, -1)
    q_filtered = q_flat[:, flat_mask]
    rho_air_flat = rho_air_trim.values.reshape(nz_trim, -1)
    rho_air_filtered = rho_air_flat[:, flat_mask]
    #sigma_num_flat = sigma_num_trim.values.reshape(nz_trim, -1)
    #sigma_num_filtered = sigma_num_flat[:, flat_mask]
    #skew_num_flat = skew_num_trim.values.reshape(nz_trim, -1)
    #skew_num_filtered = skew_num_flat[:, flat_mask]
    #kurt_num_flat = kurt_num_trim.values.reshape(nz_trim, -1)
    #kurt_num_filtered = kurt_num_flat[:, flat_mask]
    #sigma_mass_flat = sigma_mass_trim.values.reshape(nz_trim, -1)
    #sigma_mass_filtered = sigma_mass_flat[:, flat_mask]
    #skew_mass_flat = skew_mass_trim.values.reshape(nz_trim, -1)
    #skew_mass_filtered = skew_mass_flat[:, flat_mask]
    #kurt_mass_flat = kurt_mass_trim.values.reshape(nz_trim, -1)
    #kurt_mass_filtered = kurt_mass_flat[:, flat_mask]
    N_0_r_flat = N_0_r_trim.values.reshape(nz_trim, -1)
    N_0_r_filtered = N_0_r_flat[:, flat_mask]
    N_0_c_flat = N_0_c_trim.values.reshape(nz_trim, -1)
    N_0_c_filtered = N_0_c_flat[:, flat_mask]
    lambda_r_flat = lambda_r_trim.values.reshape(nz_trim, -1)
    lambda_r_filtered = lambda_r_flat[:, flat_mask]
    lambda_c_flat = lambda_c_trim.values.reshape(nz_trim, -1)
    lambda_c_filtered = lambda_c_flat[:, flat_mask]
    mu_c_flat = mu_c_trim.values.reshape(nz_trim, -1)
    mu_c_filtered = mu_c_flat[:, flat_mask]
    q_gt_100um_flat = q_gt_100um_trim.values.reshape(nz_trim, -1)
    q_gt_100um_filtered = q_gt_100um_flat[:, flat_mask]
    N_gt_100um_flat = N_gt_100um_trim.values.reshape(nz_trim, -1)
    N_gt_100um_filtered = N_gt_100um_flat[:, flat_mask]

    ctt_flat = ctt_trim.values.flatten()
    ctt_filtered = ctt_flat[flat_mask]  # Shape: (X,)
    cth_flat = cth_trim.flatten()
    cth_filtered = cth_flat[flat_mask]  
    opd_flat = opd_trim.values.flatten()
    opd_filtered = opd_flat[flat_mask]  
    lwp_flat = lwp_trim.values.flatten()
    lwp_filtered = lwp_flat[flat_mask]  
    twp_flat = twp_trim.values.flatten()
    twp_filtered = twp_flat[flat_mask] 
    iwp_flat = iwp_trim.values.flatten()
    iwp_filtered = iwp_flat[flat_mask] 
    
    #---------------------------------------------------
    # Now limit to points w/ total LWC > 1.e-3 g/m^3
    #---------------------------------------------------
    lwc_filtered = q_filtered*rho_air_filtered*1.e3 # g/m^3
    #valid_mask = (lwc_filtered > 1.e-3) & (temp_filtered >= 0.)
    #valid_mask = (lwc_filtered > 1.e-3) & (temp_filtered >= -20.)
    valid_mask = (lwc_filtered > 1.e-3) & (temp_filtered >= -8.5)
    #valid_mask = (lwc_filtered > 1.e-3)
    if np.size(valid_mask) == 0.:
        print('No congestus ID points w/ LWC > 1.e-3 g/m^3.')
        return

    lwc_valid = lwc_filtered[valid_mask]
    temp_valid = temp_filtered[valid_mask]
    RH_valid = RH_filtered[valid_mask]
    D_num_valid = D_num_filtered[valid_mask]
    D_vol_valid = D_vol_filtered[valid_mask]
    D_mass_valid = D_mass_filtered[valid_mask]
    mmd_valid = mmd_filtered[valid_mask]
    N_valid = N_filtered[valid_mask]
    q_valid = q_filtered[valid_mask]
    rho_air_valid = rho_air_filtered[valid_mask]
    #sigma_num_valid = sigma_num_filtered[valid_mask]
    #skew_num_valid = skew_num_filtered[valid_mask]
    #kurt_num_valid = kurt_num_filtered[valid_mask]
    #sigma_mass_valid = sigma_mass_filtered[valid_mask]
    #skew_mass_valid = skew_mass_filtered[valid_mask]
    #kurt_mass_valid = kurt_mass_filtered[valid_mask]
    N_0_r_valid = N_0_r_filtered[valid_mask]
    N_0_c_valid = N_0_c_filtered[valid_mask]
    lambda_r_valid = lambda_r_filtered[valid_mask]
    lambda_c_valid = lambda_c_filtered[valid_mask]
    mu_c_valid = mu_c_filtered[valid_mask]
    q_gt_100um_valid = q_gt_100um_filtered[valid_mask]
    N_gt_100um_valid = N_gt_100um_filtered[valid_mask]
    qr_valid = qr_filtered[valid_mask]
    Nr_valid = Nr_filtered[valid_mask]
    qc_valid = qc_filtered[valid_mask]
    Nc_valid = Nc_filtered[valid_mask]
    z_agl_valid = z_agl_filtered[valid_mask]
    zamsl_valid = zamsl_filtered[valid_mask]
    
    save_path = '/pscratch/sd/m/mckenna/cacti/wrf/temp_cong_dsd_params/'
    str_dt = dsd_datetime.strftime('%Y%m%d_%H:%M:%S')
    outname = save_path+"cong_dsd_params_"+str_dt+"_v2.npz"
    np.savez(outname,\
                z_agl=z_agl_valid,\
                zamsl=zamsl_valid,\
                temp=temp_valid,\
                RH=RH_valid,\
                lwc=lwc_valid,\
                D_num=D_num_valid,\
                D_vol=D_vol_valid,\
                D_mass=D_mass_valid,\
                mmd=mmd_valid,\
                N=N_valid,\
                q=q_valid,\
                rho_air=rho_air_valid,\
                #sigma_num=sigma_num_valid,\
                #skew_num=skew_num_valid,\
                #kurt_num=kurt_num_valid,\
                #sigma_mass=sigma_mass_valid,\
                #skew_mass=skew_mass_valid,\
                #kurt_mass=kurt_mass_valid,\
                N_0_r=N_0_r_valid,\
                N_0_c=N_0_c_valid,\
                lambda_r=lambda_r_valid,\
                lambda_c=lambda_c_valid,\
                mu_c=mu_c_valid,\
                q_gt_100um=q_gt_100um_valid,\
                N_gt_100um=N_gt_100um_valid,\
                qr=qr_valid,\
                qc=qc_valid,\
                Nr=Nr_valid,\
                Nc=Nc_valid,\
                )
    
    return outname

In [12]:
%%time
#dumid = 3100
for dumid in range(205,250):
    print(dumid)
    dum = driver_func(ct_files[dumid],dsd_files[dumid],wrf_3d_files)

205
CT file: /pscratch/sd/m/mckenna/cacti/wrf/derived_3km/WRF_CACTI_3km_derived_20181017_03:15:00.nc
DSD file: /pscratch/sd/m/mckenna/cacti/wrf/derived_3km_dsd_parameters/WRF_CACTI_3km_DSD_parameters_liq_only_20181017_03:15:00.nc
206
CT file: /pscratch/sd/m/mckenna/cacti/wrf/derived_3km/WRF_CACTI_3km_derived_20181017_03:30:00.nc
DSD file: /pscratch/sd/m/mckenna/cacti/wrf/derived_3km_dsd_parameters/WRF_CACTI_3km_DSD_parameters_liq_only_20181017_03:30:00.nc
207
CT file: /pscratch/sd/m/mckenna/cacti/wrf/derived_3km/WRF_CACTI_3km_derived_20181017_03:45:00.nc
DSD file: /pscratch/sd/m/mckenna/cacti/wrf/derived_3km_dsd_parameters/WRF_CACTI_3km_DSD_parameters_liq_only_20181017_03:45:00.nc
Skipping file because warmest CTT in trimmed domain is colder than -25 deg C or is NaN.
208
CT file: /pscratch/sd/m/mckenna/cacti/wrf/derived_3km/WRF_CACTI_3km_derived_20181017_04:00:00.nc
DSD file: /pscratch/sd/m/mckenna/cacti/wrf/derived_3km_dsd_parameters/WRF_CACTI_3km_DSD_parameters_liq_only_20181017_04:0

KeyboardInterrupt: 

In [ ]:
80*5

# Dask stuff

In [13]:
dask.config.config["distributed"]["dashboard"]["link"] = "{JUPYTERHUB_SERVICE_PREFIX}proxy/{host}:{port}/status" 

In [14]:
cluster = LocalCluster(n_workers=80,threads_per_worker=1,memory_limit='5GB')#,dashboard_address=':8787')
client = Client(cluster)

2026-06-26 13:18:41,011 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='jupyter.nersc.gov', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/tornado/web.py", line 1790, in _execute
    result = await result
             ^^^^^^^^^^^^
  File "/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/tornado/websocket.py", line 273, in get
    await self.ws_connection.accept_connection(self)
  File "/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/tornado/websocket.py", line 863, in accept_connection
    await self._accept_connection(handler)
  File "/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/tornado/websocket.py", line 946, in _accept_connection
    await self._receive_frame_loop()
  File "/global/c

In [15]:
cluster

LocalCluster(6a24ff7b, 'tcp://127.0.0.1:41961', workers=80, threads=80, memory=372.53 GiB)

In [16]:
def suppress_warnings():
    import warnings
    warnings.filterwarnings("ignore", message=".*All-NaN axis encountered*", category=UserWarning)
    warnings.filterwarnings("ignore", category=RuntimeWarning)

client.run(suppress_warnings)

{'tcp://127.0.0.1:33065': None,
 'tcp://127.0.0.1:33385': None,
 'tcp://127.0.0.1:33421': None,
 'tcp://127.0.0.1:33621': None,
 'tcp://127.0.0.1:33647': None,
 'tcp://127.0.0.1:34059': None,
 'tcp://127.0.0.1:34295': None,
 'tcp://127.0.0.1:34641': None,
 'tcp://127.0.0.1:34857': None,
 'tcp://127.0.0.1:34861': None,
 'tcp://127.0.0.1:35043': None,
 'tcp://127.0.0.1:35067': None,
 'tcp://127.0.0.1:35429': None,
 'tcp://127.0.0.1:35461': None,
 'tcp://127.0.0.1:35637': None,
 'tcp://127.0.0.1:35893': None,
 'tcp://127.0.0.1:36051': None,
 'tcp://127.0.0.1:36123': None,
 'tcp://127.0.0.1:36255': None,
 'tcp://127.0.0.1:36497': None,
 'tcp://127.0.0.1:36571': None,
 'tcp://127.0.0.1:36573': None,
 'tcp://127.0.0.1:36585': None,
 'tcp://127.0.0.1:36849': None,
 'tcp://127.0.0.1:37085': None,
 'tcp://127.0.0.1:37121': None,
 'tcp://127.0.0.1:37295': None,
 'tcp://127.0.0.1:37589': None,
 'tcp://127.0.0.1:37597': None,
 'tcp://127.0.0.1:37707': None,
 'tcp://127.0.0.1:38275': None,
 'tcp://

In [17]:
%%time
results = []
start_id = 0
#end_id = 250
end_id = len(ct_files)
for ii in range(start_id,end_id):
    result = dask.delayed(driver_func)(ct_files[ii],dsd_files[ii],wrf_3d_files)
    results.append(result)

CPU times: user 280 ms, sys: 91.5 ms, total: 372 ms
Wall time: 288 ms


In [18]:
%%time
futures = client.compute(results)  # results is a list of delayed or dask collections

CPU times: user 404 ms, sys: 14.5 ms, total: 418 ms
Wall time: 411 ms


/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()
/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()
/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()
/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()
/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/pyproj/__init__.py:89: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()
/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-pac

CT file: /pscratch/sd/m/mckenna/cacti/wrf/derived_3km/WRF_CACTI_3km_derived_20181108_14:00:00.nc
DSD file: /pscratch/sd/m/mckenna/cacti/wrf/derived_3km_dsd_parameters/WRF_CACTI_3km_DSD_parameters_liq_only_20181108_14:00:00.nc
Skipping file because no cloud present.
CT file: /pscratch/sd/m/mckenna/cacti/wrf/derived_3km/WRF_CACTI_3km_derived_20181125_04:00:00.nc
DSD file: /pscratch/sd/m/mckenna/cacti/wrf/derived_3km_dsd_parameters/WRF_CACTI_3km_DSD_parameters_liq_only_20181125_04:00:00.nc
CT file: /pscratch/sd/m/mckenna/cacti/wrf/derived_3km/WRF_CACTI_3km_derived_20181121_08:15:00.nc
DSD file: /pscratch/sd/m/mckenna/cacti/wrf/derived_3km_dsd_parameters/WRF_CACTI_3km_DSD_parameters_liq_only_20181121_08:15:00.nc
Skipping file because no cloud present.
CT file: /pscratch/sd/m/mckenna/cacti/wrf/derived_3km/WRF_CACTI_3km_derived_20181102_03:30:00.nc
DSD file: /pscratch/sd/m/mckenna/cacti/wrf/derived_3km_dsd_parameters/WRF_CACTI_3km_DSD_parameters_liq_only_20181102_03:30:00.nc
Skipping file be

2026-06-26 13:27:04,358 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/distributed/comm/tcp.py", line 225, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/distributed/worker.py", line 1250, in heartbeat
    response = await retry_operation(
               ^^^^^^^^^^^^^^^^^^^^^^
  File "/global/common/software/m1657/mckenna/flextrkr/lib/python3.11/site-packages/distributed/utils_comm.py", line 459, in retry_operation
    return await retry(
           ^^^^^^^^^^^^
  File "/global/common/software/m1657/m

In [ ]:
%%time
print('Step 1')
# Step 1: List .npz files
npz_files = sorted(glob.glob('/pscratch/sd/m/mckenna/cacti/wrf/temp_cong_dsd_params/cong_dsd_params_*.npz'))
print('Step 2')
# Step 2: Get all keys by inspecting the first file
with np.load(npz_files[0]) as sample:
    all_keys = list(sample.keys())
print('Step 3')
# Step 3: Delayed function to load a single file
@delayed
def load_npz_var(filepath, varname):
    with np.load(filepath) as data:
        return data[varname]
print('Step 4')
# Step 4: For each variable, collect delayed arrays and concatenate
def extract_and_concat(varname, file_list):
    delayed_arrays = [load_npz_var(f, varname) for f in file_list]
    return delayed(np.concatenate)(delayed_arrays)
print('Step 5')
# Step 5: Build delayed concatenation for each variable
concatenated_vars = {
    k: extract_and_concat(k, npz_files) for k in all_keys
}
print('Step 6')
# Step 6: Compute in parallel
computed_values = compute(*concatenated_vars.values())
print('Step 7')

# Step 7: Reconstruct dictionary
final_data = dict(zip(concatenated_vars.keys(), computed_values))

In [ ]:
for key,val in final_data.items():
    print(key,np.shape(val),np.nanmax(val),np.nanmin(val))

In [ ]:
fig = plt.figure(figsize=(12,8),constrained_layout=True)
ax1 = fig.add_subplot(221)
ax2 = fig.add_subplot(222)
ax3 = fig.add_subplot(223)
ax4 = fig.add_subplot(224)
Fontsize=11
axlist = [ax1,ax2,ax3,ax4]
for ax in axlist:
    ax.grid(which='both',lw=1,c='dimgrey',ls='dotted')
    ax.tick_params(labelsize=Fontsize)

ax1.hist(ctt_filtered,bins=10,histtype='step',lw=2,color='k',density=True)
ax1.set_xlabel('CTT [$^{\\circ}$C]',fontsize=Fontsize)
ax1.set_ylabel('Probability Density',fontsize=Fontsize)

ax2.hist(cth_filtered*1.e-3,bins=10,histtype='step',lw=2,color='k',density=True)
ax2.set_xlabel('CTH [km]',fontsize=Fontsize)
ax2.set_ylabel('Probability Density',fontsize=Fontsize)

mean_z = np.mean(zamsl_filtered,axis=1)
lwc = qc_filtered*rho_air_filtered*1.e3
clwc = qc_filtered*rho_air_filtered*1.e3
rwc = qr_filtered*rho_air_filtered*1.e3
#lwc[lwc == 0.] = np.nan
mean_lwc = np.nanmean(lwc,axis=1)
mean_clwc = np.nanmean(clwc,axis=1)
mean_rwc = np.nanmean(rwc,axis=1)
ax3.plot(mean_lwc,mean_z*1.e-3,lw=2,c='k',label='Total')
ax3.plot(mean_clwc,mean_z*1.e-3,lw=1.5,c='blue',label='Cloud')
ax3.plot(mean_rwc,mean_z*1.e-3,lw=1.5,c='red',label='Rain')
ax3.set_ylabel('Height [km]',fontsize=Fontsize)
ax3.set_xlabel('Mean LWC [g m$^{-3}$]',fontsize=Fontsize)
ax3.set_ylim(0,5)
ax3.legend(fontsize=Fontsize,loc='best')

mean_D_num = np.nanmean(D_num_filtered*1.e6,axis=1)
mean_D_mass = np.nanmean(D_mass_filtered*1.e6,axis=1)
mean_mmd = np.nanmean(mmd_filtered*1.e6,axis=1)
ax4.plot(mean_D_num,mean_z*1.e-3,lw=2,c='k',label='NWMD')
ax4.plot(mean_D_mass,mean_z*1.e-3,lw=2,c='blue',label='MWMD')
ax4.plot(mean_mmd,mean_z*1.e-3,lw=2,c='red',label='MMD')
ax4.set_ylabel('Height [km]',fontsize=Fontsize)
ax4.set_xlabel('Diameter [$\\mu$m]',fontsize=Fontsize)
ax4.set_ylim(0,5)
ax4.legend(fontsize=Fontsize,loc='best')

ax3.set_xscale('log')
plt.show()
plt.close()

In [ ]:
fig = plt.figure(figsize=(8,6))
ax1 = fig.add_subplot(111)
Fontsize=11
axlist = [ax1]
for ax in axlist:
    ax.tick_params(labelsize=Fontsize)
ax1.contour(lon,lat,ter_hgt,lw=1,colors=['black','dimgrey','grey','lightgrey'],levels=[1000.,1500.,2000.,2500.])
#tmp_plot = ax1.contourf(lon,lat,ctt,levels=np.arange(-25,11,1),cmap='nipy_spectral',extend='both')
tmp_plot = ax1.pcolormesh(lon,lat,ctt,vmin=-5,vmax=5,cmap='RdYlBu_r')
cbar = fig.colorbar(tmp_plot,extend='both')
cbar.ax.set_ylabel('CTT [$^{\\circ}$C]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

# Top edge
top_lon = lon_trim[0, :]
top_lat = lat_trim[0, :]

# Bottom edge
bot_lon = lon_trim[-1, :]
bot_lat = lat_trim[-1, :]

# Left edge
left_lon = lon_trim[:, 0]
left_lat = lat_trim[:, 0]

# Right edge
right_lon = lon_trim[:, -1]
right_lat = lat_trim[:, -1]


bbox_lon = np.concatenate([top_lon, right_lon, bot_lon[::-1], left_lon[::-1]])
bbox_lat = np.concatenate([top_lat, right_lat, bot_lat[::-1], left_lat[::-1]])

ax1.plot(bbox_lon, bbox_lat, c='red', lw=2, label='Trim Box')
ax1.legend(fontsize=Fontsize)

plt.show()
plt.close()

In [ ]:
max_mmd = np.nanmax(mmd,axis=(0))
max_D_num = np.nanmax(D_num,axis=(0))
mean_D_num = np.nanmean(D_num,axis=(0))

#csapr_nan_id = np.where(np.isnan(col_max_ref))
#mean_D_num[csapr_nan_id[0],csapr_nan_id[1]] = np.nan

fig = plt.figure(figsize=(8,6))
ax1 = fig.add_subplot(111)
Fontsize=11
axlist = [ax1]
for ax in axlist:
    ax.tick_params(labelsize=Fontsize)
ax1.contour(lon,lat,ter_hgt,lw=1,colors=['black','dimgrey','grey','lightgrey'],levels=[1000.,1500.,2000.,2500.])
#tmp_plot = ax1.pcolormesh(lon,lat,max_mmd*1.e6,cmap='turbo',norm=mpl.colors.LogNorm(vmin=1,vmax=5.e3))
#tmp_plot = ax1.pcolormesh(lon,lat,max_D_num*1.e6,cmap='turbo',vmin=1,vmax=500)
tmp_plot = ax1.pcolormesh(lon,lat,mean_D_num*1.e6,cmap='turbo',vmin=1,vmax=300)
cbar = fig.colorbar(tmp_plot,extend='max')
#cbar.ax.set_ylabel('Col. Max. MMD [$\\mu$m]',fontsize=Fontsize)
#cbar.ax.set_ylabel('Col. Max. NWMD [$\\mu$m]',fontsize=Fontsize)
cbar.ax.set_ylabel('Vertical Avg. NWMD [$\\mu$m]',fontsize=Fontsize)
cbar.ax.tick_params(labelsize=Fontsize)

# Top edge
top_lon = lon_trim[0, :]
top_lat = lat_trim[0, :]

# Bottom edge
bot_lon = lon_trim[-1, :]
bot_lat = lat_trim[-1, :]

# Left edge
left_lon = lon_trim[:, 0]
left_lat = lat_trim[:, 0]

# Right edge
right_lon = lon_trim[:, -1]
right_lat = lat_trim[:, -1]


bbox_lon = np.concatenate([top_lon, right_lon, bot_lon[::-1], left_lon[::-1]])
bbox_lat = np.concatenate([top_lat, right_lat, bot_lat[::-1], left_lat[::-1]])

ax1.plot(bbox_lon, bbox_lat, c='red', lw=2, label='Trim Box')
ax1.legend(fontsize=Fontsize)

plt.show()
plt.close()

In [ ]:
ds_dsd